# AI4BCI: FBCSP-style baseline under the existing session split

**Run top to bottom in Google Colab or Jupyter. A CPU is sufficient.**
This notebook downloads the public MATLAB files, fits the baseline, and includes the
original EEGNet scores for a paired comparison. No other project files are required.
The first run downloads 18 recordings; subsequent runs reuse verified cached files.
On Colab, the configuration below mounts Google Drive to preserve downloads and results.

This is an additional **post-study exploratory baseline**, not a new primary SAST analysis.
No new manuscript score is assumed until all nine participants finish.
Use one runtime per study folder. Do not change the pipeline in response to session-2
results and then present those same trials as an untouched test.

## 1. Install dependencies
If Jupyter asks for a kernel restart after installation, restart and continue from the configuration cell.

In [ ]:
import importlib.metadata, importlib.util, subprocess, sys
required = {"mne": "1.10.2", "scikit-learn": "1.7.2"}
needed=[]
for name, version in required.items():
    try: same = importlib.metadata.version(name) == version
    except importlib.metadata.PackageNotFoundError: same = False
    if not same: needed.append(f"{name}=={version}")
for name in ("numpy", "scipy", "pandas", "matplotlib", "joblib", "threadpoolctl"):
    try: importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError: needed.append(name)
if needed:
    loaded = [name for name in ("mne", "sklearn", "numpy", "scipy", "pandas") if name in sys.modules]
    subprocess.run([sys.executable, "-m", "pip", "install", *needed], check=True)
    if loaded: raise RuntimeError("Packages changed: restart the kernel and Run all.")


## 2. Choose storage
Keep all nine participants for the paper comparison. A smaller subset is a diagnostic run
and is explicitly marked provisional. To reuse the original raw recordings, point CACHE_DIR
at their folder; filenames must be A01T.mat, A01E.mat, …, A09E.mat.
The baseline writes to its own results directory without changing the existing study outputs.

In [ ]:
from pathlib import Path
import os
USE_GOOGLE_DRIVE = True
SUBJECTS = tuple(range(1, 10))
try:
    from google.colab import drive
except ImportError:
    drive = None
if drive is not None and USE_GOOGLE_DRIVE:
    drive.mount("/content/drive")
    WORK_DIR = Path("/content/drive/MyDrive/AI4BCI_FBCSP")
else:
    WORK_DIR = Path(os.environ.get("AI4BCI_FBCSP_DIR", "ai4bci_fbcsp_runs")).resolve()
CACHE_DIR = WORK_DIR / "raw_data"
WORK_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
print("Working directory:", WORK_DIR)


## 3. Fixed protocol and method

| Item | Setting |
|---|---|
| Dataset | BCI IV-2a / BNCI2014-001; nine participants; four classes; 22 EEG channels |
| Fit | Session 1 (T), runs 1–5: 240 trials per participant |
| Select | Session 1, run 6: 48 trials; four candidate feature counts |
| Test | All session 2 (E): 288 trials; no fitting or selection |
| Trial retention | All artifact-marked trials retained; artifact-free score is secondary |
| Common front end | Same original loader: fourth-order 4–38 Hz Butterworth filtering forward/backward within each run; 0–4 s after cue; polyphase 250→125 Hz; 22×500 samples |
| Scaling | Channel mean/std from the 240 training trials only |
| Filter bank | 4–8, 8–12, 12–16, 16–20, 20–24, 24–28, 28–32, 32–36, 34–38 Hz |
| Spatial features | Four one-vs-rest CSP models per band; four extreme components; fixed covariance shrinkage 0.1; log mean-square power |
| Feature selection | Training-only mutual information; fixed random seed; select 8, 16, 32, or 64 of 144 features |
| Classifier | Multiclass LDA with automatic covariance shrinkage |
| Selection | Highest validation BA, then lowest log loss, then candidate order |
| Final fit | Runs 1–5 only; no refit on validation or target trials |
| Uncertainty | 10,000 participant bootstrap resamples; paired FBCSP-minus-EEGNet contrast |

**Name this method “FBCSP-style (MI + shrinkage LDA)” in the paper.**
It follows Ang et al.'s filter-bank/spatial-filter/feature-selection structure.
The competition method used causal Chebyshev filters, different feature-selection details,
and a Naïve Bayesian Parzen Window classifier. This notebook uses Butterworth filters,
scikit-learn mutual information and shrinkage LDA; it is not an exact reproduction.
The ninth 4-Hz band overlaps the preceding band to stay inside the fixed 4–38 Hz front end.
Additional subband filtering is forward/backward within each extracted epoch at 125 Hz.
This and the common front end make the comparison offline.

There is one deterministic baseline fit per participant. The original EEGNet's five seeds
are averaged within each participant before comparison. All source choices are locked
before any session-2 file is opened.

Sources: [Ang et al. (2012)](https://doi.org/10.3389/fnins.2012.00039),
[dataset description](https://www.bbci.de/competition/iv/desc_2a.pdf),
[MNE CSP](https://mne.tools/1.10/generated/mne.decoding.CSP.html),
[scikit-learn LDA](https://scikit-learn.org/stable/modules/generated/sklearn.discriminant_analysis.LinearDiscriminantAnalysis.html).

**Performance:** selecting a GPU runtime does not move this MNE CSP / scikit-learn shrinkage-LDA pipeline to CUDA. Keep a CPU runtime. The source scaler and mutual-information ranking are now fitted once and reused across the four candidate feature counts; the candidate set, selection rule, and predictions are unchanged. Source fitting prints its elapsed CPU time. See the [supported Array API estimators](https://scikit-learn.org/1.7/modules/array_api.html): LDA's GPU-compatible solver is SVD, while this baseline uses LSQR with shrinkage.


## 4. Self-contained experiment engine

Download, parsing and common preprocessing functions are copied from the original
ai4bci_all_experiments notebook. The additional CSP, selection and reporting code
runs independently. Results record the full engine, its SHA-256, software versions,
raw-file checksums, selected settings and trial predictions.
Changing the engine, software, configuration or reference creates a distinct run folder.

In [ ]:
ENGINE_SOURCE = '\n\n\nfrom pathlib import Path\nfrom functools import partial\nfrom copy import deepcopy\nimport hashlib\nimport importlib.metadata\nimport io\nimport json\nimport os\nimport pickle\nimport platform\nimport shutil\nimport time\nimport urllib.request\nimport zipfile\n\nimport joblib\nimport matplotlib.pyplot as plt\nimport mne\nimport numpy as np\nimport pandas as pd\nimport scipy\nimport sklearn\nfrom scipy.io import loadmat\nfrom scipy.signal import butter, resample_poly, sosfiltfilt\nfrom mne.decoding import CSP\nfrom sklearn.discriminant_analysis import LinearDiscriminantAnalysis\nfrom sklearn.feature_selection import SelectKBest, mutual_info_classif\nfrom sklearn.metrics import balanced_accuracy_score, log_loss\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler\nfrom threadpoolctl import threadpool_limits\n\nmne.set_log_level("ERROR")\nfor module, distribution in ((np, "numpy"), (scipy, "scipy"), (pd, "pandas"),\n                             (mne, "mne"), (sklearn, "scikit-learn")):\n    if module.__version__ != importlib.metadata.version(distribution):\n        raise RuntimeError("A loaded package changed during installation. Restart the kernel, then run from section 2.")\n\n\ndef atomic_json(path, value):\n    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(path.suffix + ".tmp")\n    tmp.write_text(json.dumps(value, indent=2, sort_keys=True, allow_nan=False))\n    os.replace(tmp, path)\n\ndef sha_file(path):\n    h = hashlib.sha256()\n    with open(path, "rb") as f:\n        for chunk in iter(lambda: f.read(1024 * 1024), b""): h.update(chunk)\n    return h.hexdigest()\n\ndef download_session(subject, session, cache):\n    name = f"A{subject:02d}{session}.mat"\n    path = Path(cache) / name; path.parent.mkdir(parents=True, exist_ok=True)\n    record = path.with_suffix(".sha256.json")\n    url = "https://lampx.tugraz.at/~bci/database/001-2014/" + name\n    if path.exists() and record.exists():\n        info = json.loads(record.read_text())\n        if sha_file(path) != info["sha256"]:\n            raise RuntimeError(f"Corrupt cached download: {path}; move it aside and rerun.")\n        return path\n    if not path.exists():\n        for attempt in range(3):\n            tmp = path.with_suffix(".part")\n            try:\n                request = urllib.request.Request(url, headers={"User-Agent": "AI4BCI-reproduction/1"})\n                with urllib.request.urlopen(request, timeout=90) as response, tmp.open("wb") as out:\n                    expected = response.headers.get("Content-Length")\n                    shutil.copyfileobj(response, out, length=1024 * 1024)\n                if expected and tmp.stat().st_size != int(expected):\n                    raise IOError("Incomplete dataset download")\n                loadmat(tmp, variable_names=["data"])\n                os.replace(tmp, path); break\n            except Exception:\n                if attempt == 2: raise\n                time.sleep(2 ** attempt)\n    loadmat(path, variable_names=["data"])\n    atomic_json(record, {"url": url, "sha256": sha_file(path), "bytes": path.stat().st_size})\n    return path\n\ndef read_session(path, subject, session):\n    runs = np.atleast_1d(loadmat(path, struct_as_record=False, squeeze_me=True)["data"])\n    arrays, rows = [], []\n    sos = butter(4, [4, 38], btype="bandpass", fs=250, output="sos")\n    run_id = 0\n    for raw_id, run in enumerate(runs):\n        markers = np.atleast_1d(run.trial).astype(int)\n        if not markers.size: continue  # Initial EOG/baseline recordings are not task runs.\n        run_id += 1\n        labels = np.atleast_1d(run.y).astype(int) - 1\n        flags = np.atleast_1d(run.artifacts).astype(int)\n        x = np.asarray(run.X, dtype=np.float64)\n        if float(run.fs) != 250 or x.ndim != 2 or x.shape[1] != 25:\n            raise ValueError("Unexpected sampling rate or channel count")\n        if len(markers) != 48 or len(flags) != 48 or not np.array_equal(np.bincount(labels, minlength=4), [12]*4):\n            raise ValueError("Unexpected trial/class/artifact counts; do not silently drop trials")\n        if not np.isfinite(x[:, :22]).all():\n            raise ValueError(f"Nonfinite EEG in {path}, task run {run_id}; inspect before analysis")\n        filtered = sosfiltfilt(sos, x[:, :22], axis=0, padtype="odd")\n        for trial, (marker, label, artifact) in enumerate(zip(markers, labels, flags)):\n            start = int(marker) - 1 + 500  # MATLAB one-based trial start + 2 s to cue.\n            epoch = filtered[start:start + 1000]\n            if epoch.shape != (1000, 22): raise ValueError("Incomplete 0--4 s cue epoch")\n            epoch = resample_poly(epoch, 1, 2, axis=0, window=("kaiser", 5.), padtype="constant")\n            arrays.append(epoch.T.astype(np.float32))\n            rows.append(dict(subject=subject, session=session, run=run_id, raw_run=raw_id,\n                             trial=trial, trial_start_sample=int(marker)-1, cue_sample=start,\n                             label=int(label), artifact=int(artifact),\n                             trial_id=f"A{subject:02d}{session}-r{run_id}-t{trial:02d}"))\n    if run_id != 6: raise ValueError(f"Expected six task runs; found {run_id}")\n    return np.stack(arrays), pd.DataFrame(rows)\n\nBANDS = [(4, 8), (8, 12), (12, 16), (16, 20), (20, 24),\n         (24, 28), (28, 32), (32, 36), (34, 38)]\nFEATURE_COUNTS = (8, 16, 32, 64)\n\n\ndef band_epochs(x, band):\n    if x.ndim != 3 or x.shape[1:] != (22, 500) or not np.isfinite(x).all():\n        raise ValueError("Expected finite trial × 22 EEG channels × 500 samples")\n    sos = butter(4, band, btype="bandpass", fs=125, output="sos")\n    return sosfiltfilt(sos, x.astype(np.float64), axis=-1, padtype="odd")\n\n\ndef fit_spatial(x, y, bands=BANDS):\n    if not np.array_equal(np.unique(y), np.arange(4)) or len(x) != len(y):\n        raise ValueError("Training data must contain all four classes")\n    models, features = [], []\n    for band in bands:\n        xb = band_epochs(x, band)\n        band_models = []\n        for label in range(4):\n            csp = CSP(n_components=4, reg=0.1, log=True, norm_trace=False,\n                      cov_est="concat", component_order="alternate", rank="full")\n            features.append(csp.fit_transform(xb, (y == label).astype(int)))\n            band_models.append(csp)\n        models.append(band_models)\n    return np.concatenate(features, axis=1), models\n\n\ndef transform_spatial(x, models, bands=BANDS):\n    if len(models) != len(bands):\n        raise ValueError("Filter-bank model does not match the prescribed bands")\n    features = []\n    for band, band_models in zip(bands, models):\n        xb = band_epochs(x, band)\n        features.extend(csp.transform(xb) for csp in band_models)\n    result = np.concatenate(features, axis=1)\n    if not np.isfinite(result).all():\n        raise ValueError("Nonfinite CSP features")\n    return result\n\n\ndef choose_head(train_features, train_y, val_features, val_y, counts=FEATURE_COUNTS):\n    # These source-only fits are identical for every candidate feature count.\n    scaler = StandardScaler().fit(train_features)\n    train_scaled = scaler.transform(train_features)\n    ranking = SelectKBest(\n        partial(mutual_info_classif, random_state=20261012), k="all"\n    ).fit(train_scaled, train_y)\n    candidates, fitted = [], []\n    for order, count in enumerate(counts):\n        if not 0 < count <= train_features.shape[1]:\n            raise ValueError("Selected-feature count exceeds the feature bank")\n        selector = deepcopy(ranking).set_params(k=count)\n        classifier = LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")\n        classifier.fit(selector.transform(train_scaled), train_y)\n        model = make_pipeline(scaler, selector, classifier)\n        probabilities = model.predict_proba(val_features)\n        ba = 100 * balanced_accuracy_score(val_y, model.predict(val_features))\n        loss = log_loss(val_y, probabilities, labels=np.arange(4))\n        candidates.append(dict(order=order, selected_features=count,\n                               validation_ba=float(ba), validation_log_loss=float(loss)))\n        fitted.append(model)\n    best = min(range(len(candidates)),\n               key=lambda i: (-candidates[i]["validation_ba"],\n                              candidates[i]["validation_log_loss"], i))\n    return fitted[best], candidates, best\n\n\ndef create_study(base, subjects, engine_source, reference_csv):\n    subjects = tuple(subjects)\n    if not subjects or len(set(subjects)) != len(subjects) or not set(subjects) <= set(range(1, 10)):\n        raise ValueError("SUBJECTS must be unique integers from 1 through 9")\n    config = dict(subjects=subjects, bands=BANDS, feature_counts=FEATURE_COUNTS,\n                  csp_components=4, csp_shrinkage=0.1, feature_selection_seed=20261012,\n                  bootstrap_seed=20261012, bootstrap_samples=10000,\n                  classifier="multiclass LDA, lsqr solver, automatic shrinkage",\n                  front_end="Original run-level 4–38 Hz / cue 0–4 s / 125 Hz; source scaling",\n                  filter_bank="Fourth-order Butterworth, forward/backward within each epoch",\n                  split="Session 1 runs 1–5 fit; run 6 select; all session 2 test",\n                  tuning="Four feature counts; no refit on validation; no target fitting",\n                  status="Post-study exploratory external baseline, not a new primary contrast")\n    versions = {name: importlib.metadata.version(name)\n                for name in ("numpy", "scipy", "pandas", "mne", "scikit-learn")}\n    identity = dict(config=config, software=versions, python=platform.python_version(),\n                    engine_sha256=hashlib.sha256(engine_source.encode()).hexdigest(),\n                    reference_sha256=hashlib.sha256(reference_csv.encode()).hexdigest())\n    study_id = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()[:16]\n    root = Path(base) / f"FBCSP_LDA_{study_id}"\n    root.mkdir(parents=True, exist_ok=True)\n    atomic_json(root / "protocol.json", identity)\n    (root / "fbcsp_engine.py").write_text(engine_source)\n    (root / "original_eegnet_per_seed.csv").write_text(reference_csv)\n    return root\n\n\ndef select_all(subjects, cache, root):\n    selections = []\n    for subject in subjects:\n        directory = root / f"subject_{subject:02d}"\n        directory.mkdir(exist_ok=True)\n        path = download_session(subject, "T", cache)\n        source_sha = sha_file(path)\n        model_path, selection_path = directory / "model.joblib", directory / "selection.json"\n        if model_path.exists() and selection_path.exists():\n            selection = json.loads(selection_path.read_text())\n            if selection["source_sha256"] != source_sha or selection["model_sha256"] != sha_file(model_path):\n                raise RuntimeError("Cached model/data hash mismatch; inspect the study directory")\n            print(f"Participant {subject}: using completed source selection", flush=True)\n        else:\n            x, frame = read_session(path, subject, "T")\n            train = frame.run.to_numpy() <= 5\n            if train.sum() != 240 or (~train).sum() != 48:\n                raise ValueError("Unexpected source train/validation split")\n            mean = x[train].mean(axis=(0, 2), keepdims=True, dtype=np.float64).astype(np.float32)\n            scale = x[train].std(axis=(0, 2), keepdims=True, dtype=np.float64).astype(np.float32)\n            scale = np.maximum(scale, 1e-6)\n            x = ((x - mean) / scale).astype(np.float32)\n            y = frame.label.to_numpy()\n            print(f"Participant {subject}: fitting source-only CSP and four validation candidates", flush=True)\n            fit_started = time.perf_counter()\n            with threadpool_limits(limits=2):\n                train_features, spatial = fit_spatial(x[train], y[train])\n                val_features = transform_spatial(x[~train], spatial)\n                head, candidates, best = choose_head(\n                    train_features, y[train], val_features, y[~train])\n            bundle = dict(mean=mean, scale=scale, spatial=spatial, head=head)\n            temporary = model_path.with_suffix(".tmp")\n            joblib.dump(bundle, temporary, compress=3)\n            os.replace(temporary, model_path)\n            selection = dict(subject=int(subject), source_sha256=source_sha,\n                             model_sha256=sha_file(model_path),\n                             selected_features=candidates[best]["selected_features"],\n                             validation_ba=candidates[best]["validation_ba"],\n                             fit_seconds=time.perf_counter() - fit_started,\n                             candidates=candidates)\n            atomic_json(selection_path, selection)\n            print(f"Participant {subject}: fit/search finished in {selection[\'fit_seconds\']:.2f} s (CPU; excludes download/preprocessing)", flush=True)\n            frame.assign(split=np.where(train, "train", "validation")).to_csv(\n                directory / "source_trials.csv", index=False)\n        selections.append(selection)\n    # No evaluation-session file is opened until every participant\'s choice is locked.\n    atomic_json(root / "locked_selection.json", dict(subjects=list(subjects), selections=selections))\n    print("All source choices locked. Session 2 may now be evaluated.", flush=True)\n\n\ndef evaluate_all(subjects, cache, root):\n    lock_path = root / "locked_selection.json"\n    lock_sha = sha_file(lock_path)\n    lock = json.loads(lock_path.read_text())\n    if lock["subjects"] != list(subjects):\n        raise ValueError("Locked selections do not cover the configured participants")\n    rows = []\n    for selection in lock["selections"]:\n        subject = selection["subject"]\n        directory = root / f"subject_{subject:02d}"\n        model_path = directory / "model.joblib"\n        if sha_file(model_path) != selection["model_sha256"]:\n            raise RuntimeError("Source-fitted model changed after selection")\n        model = joblib.load(model_path)  # Only this run\'s locally created model is loaded.\n        path = download_session(subject, "E", cache)\n        x, frame = read_session(path, subject, "E")\n        x = ((x - model["mean"]) / model["scale"]).astype(np.float32)\n        with threadpool_limits(limits=2):\n            features = transform_spatial(x, model["spatial"])\n            predictions = model["head"].predict(features)\n        y = frame.label.to_numpy()\n        ba = 100 * balanced_accuracy_score(y, predictions)\n        clean = frame.artifact.to_numpy() == 0\n        clean_ba = (100 * balanced_accuracy_score(y[clean], predictions[clean])\n                    if np.array_equal(np.unique(y[clean]), np.arange(4)) else np.nan)\n        frame.assign(predicted_label=predictions, correct=predictions == y).to_csv(\n            directory / "test_predictions.csv", index=False)\n        atomic_json(directory / "target_data.json",\n                    dict(target_sha256=sha_file(path), model_sha256=selection["model_sha256"],\n                         locked_selection_sha256=lock_sha, n_trials=len(frame)))\n        rows.append(dict(subject=subject, ba=float(ba), artifact_free_ba=float(clean_ba),\n                         validation_ba=selection["validation_ba"],\n                         selected_features=selection["selected_features"]))\n        print(f"Participant {subject}: session-2 BA {ba:.2f}%", flush=True)\n    if sha_file(lock_path) != lock_sha:\n        raise RuntimeError("Selections changed during target evaluation")\n    scores = pd.DataFrame(rows)\n    scores.to_csv(root / "participant_scores.csv", index=False)\n    return scores\n\n\ndef participant_interval(values):\n    values = np.asarray(values, dtype=float)\n    if values.ndim != 1 or not len(values) or not np.isfinite(values).all():\n        raise ValueError("Intervals require finite participant-level values")\n    if len(values) < 2:\n        return float(values.mean()), np.nan, np.nan\n    rng = np.random.default_rng(20261012)\n    samples = values[rng.integers(len(values), size=(10000, len(values)))].mean(axis=1)\n    return float(values.mean()), *np.quantile(samples, [0.025, 0.975]).tolist()\n\n\ndef report_results(scores, reference_csv, root):\n    reference = pd.read_csv(io.StringIO(reference_csv))\n    if (len(reference) != 45 or reference.duplicated(["subject", "seed"]).any()\n            or set(reference.subject) != set(range(1, 10))\n            or set(reference.seed) != set(range(5))\n            or not np.isfinite(reference.ba).all()\n            or not reference.ba.between(0, 100).all()):\n        raise ValueError("Bundled EEGNet reference must contain nine participants × five seeds")\n    eegnet = reference.groupby("subject", as_index=False).ba.mean().rename(columns={"ba": "eegnet_ba"})\n    paired = scores.merge(eegnet, on="subject", validate="one_to_one").sort_values("subject").reset_index(drop=True)\n    paired["fbcsp_minus_eegnet_pp"] = paired.ba - paired.eegnet_ba\n    paired.to_csv(root / "paired_eegnet_comparison.csv", index=False)\n    summary = []\n    for label, values in (\n        ("FBCSP-style / shrinkage LDA", paired.ba),\n        ("EEGNet (original five-seed mean)", paired.eegnet_ba),\n        ("FBCSP minus EEGNet (percentage points)", paired.fbcsp_minus_eegnet_pp),\n    ):\n        mean, low, high = participant_interval(values)\n        summary.append(dict(comparison=label, mean=mean, low=low, high=high,\n                            n_subjects=len(paired), complete_nine_subjects=len(paired) == 9))\n    summary = pd.DataFrame(summary)\n    summary.to_csv(root / "summary.csv", index=False)\n    if len(paired) == 9:\n        mean, low, high = participant_interval(paired.ba)\n        (root / "paper_table_row.txt").write_text(\n            f"FBCSP-style (MI + LDA) & {mean:.1f} [{low:.1f}, {high:.1f}] & -- & -- \\\\\\\\\\n")\n    else:\n        print("PROVISIONAL: fewer than nine participants; do not use this subset as the paper result.")\n    fig, ax = plt.subplots(figsize=(9, 4))\n    ax.plot(paired.subject, paired.ba, "o-", label="FBCSP-style / shrinkage LDA")\n    ax.plot(paired.subject, paired.eegnet_ba, "s--", label="EEGNet (original seed mean)")\n    ax.axhline(25, color="grey", linestyle=":", label="Chance (25%)")\n    ax.set(xlabel="Participant", ylabel="Session-2 balanced accuracy (%)",\n           xticks=paired.subject, ylim=(0, 100))\n    ax.legend()\n    fig.tight_layout()\n    fig.savefig(root / "participant_comparison.png", dpi=180)\n    plt.show()\n    plt.close(fig)\n    archive = root.with_suffix(".zip")\n    with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:\n        for path in sorted(root.rglob("*")):\n            if path.is_file():\n                z.write(path, path.relative_to(root.parent))\n    return paired, summary, archive\n\n\ndef self_check():\n    rng = np.random.default_rng(42)\n    def synthetic(n_each):\n        y = np.repeat(np.arange(4), n_each)\n        rng.shuffle(y)\n        x = rng.normal(0, 0.5, (len(y), 22, 500))\n        wave = np.sin(2 * np.pi * 10 * np.arange(500) / 125)\n        for i, label in enumerate(y):\n            x[i, label * 4:label * 4 + 3] += 3 * wave\n        return x.astype(np.float32), y\n    train, train_y = synthetic(16)\n    val, val_y = synthetic(8)\n    test, test_y = synthetic(8)\n    train_copy = train.copy()\n    small_bank = [(8, 12)]\n    ft, spatial = fit_spatial(train, train_y, small_bank)\n    assert np.array_equal(train, train_copy), "CSP altered the input epochs"\n    fv = transform_spatial(val, spatial, small_bank)\n    head, candidates, best = choose_head(ft, train_y, fv, val_y, counts=(4, 8))\n    # Regression: cached ranking must match independently fitted sklearn pipelines.\n    for candidate in candidates:\n        reference = make_pipeline(\n            StandardScaler(),\n            SelectKBest(partial(mutual_info_classif, random_state=20261012),\n                        k=candidate["selected_features"]),\n            LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto"),\n        ).fit(ft, train_y)\n        assert candidate["validation_ba"] == 100 * balanced_accuracy_score(val_y, reference.predict(fv))\n        assert candidate["validation_log_loss"] == log_loss(val_y, reference.predict_proba(fv), labels=np.arange(4))\n        if candidate == candidates[best]:\n            np.testing.assert_array_equal(reference.predict_proba(fv), head.predict_proba(fv))\n    before = hashlib.sha256(pickle.dumps((spatial, head))).hexdigest()\n    predictions = head.predict(transform_spatial(test, spatial, small_bank))\n    assert 100 * balanced_accuracy_score(test_y, predictions) >= 75\n    assert before == hashlib.sha256(pickle.dumps((spatial, head))).hexdigest(), "Inference refitted a model"\n    assert len(candidates) == 2 and candidates[best]["selected_features"] in (4, 8)\n    assert participant_interval([1, 2, 3])[0] == 2\n    print("Synthetic fitting, held-out prediction, input immutability and interval checks passed.")\n\n\n\n'
exec(compile(ENGINE_SOURCE, "fbcsp_engine.py", "exec"))

## 5. Bundled original EEGNet reference

These are the 45 actual EEGNet session-2 records from results/e1_per_seed.csv, not examples.
Source file SHA-256: fe558c8bc773dc35a049f71a43888aab77d4641e153ef812368c686041234ca2.
Original run: BNCI2014_001_ccebe0c6749374ab (13 October 2026).
Only EEGNet / continuous-mode rows are embedded, so no original study folder is needed.

In [ ]:
REFERENCE_CSV = '''
subject,seed,ba
1,0,68.40277777777779
1,1,68.05555555555556
1,2,67.36111111111111
1,3,68.05555555555556
1,4,67.0138888888889
2,0,44.44444444444444
2,1,40.625
2,2,45.833333333333336
2,3,38.19444444444444
2,4,43.75
3,0,69.09722222222221
3,1,72.56944444444444
3,2,69.44444444444444
3,3,70.48611111111111
3,4,70.83333333333334
4,0,38.88888888888889
4,1,42.01388888888889
4,2,42.70833333333333
4,3,40.97222222222222
4,4,41.66666666666667
5,0,35.416666666666664
5,1,30.55555555555556
5,2,29.861111111111114
5,3,29.51388888888889
5,4,30.208333333333336
6,0,41.31944444444444
6,1,27.43055555555556
6,2,39.23611111111111
6,3,38.19444444444444
6,4,27.43055555555556
7,0,57.98611111111112
7,1,59.375
7,2,61.458333333333336
7,3,62.5
7,4,48.958333333333336
8,0,60.06944444444444
8,1,71.18055555555556
8,2,68.40277777777779
8,3,63.19444444444443
8,4,71.875
9,0,73.61111111111111
9,1,73.61111111111111
9,2,77.43055555555556
9,3,70.83333333333334
9,4,76.3888888888889
'''.lstrip()
print('Bundled EEGNet records:', len(pd.read_csv(io.StringIO(REFERENCE_CSV))))

## 6. Check before downloading EEG
Synthetic checks do not contribute to the reported baseline. They test held-out spatial
discrimination, inference without refitting, cue/run accounting and malformed metadata.

In [ ]:
from tempfile import TemporaryDirectory
from scipy.io import savemat

with threadpool_limits(limits=2):
    self_check()
rng = np.random.default_rng(7)
recording = rng.normal(size=(2500, 25))
task = dict(X=recording, fs=250, trial=1 + 20 * np.arange(48),
            y=np.tile(np.arange(1, 5), 12), artifacts=(np.arange(48) % 11 == 0).astype(int))
runs = np.empty((6,), dtype=object)
for i in range(6):
    runs[i] = task.copy()
with TemporaryDirectory() as temporary:
    fixture = Path(temporary) / "fixture.mat"
    savemat(fixture, {"data": runs}, do_compression=True)
    x, meta = read_session(fixture, 1, "T")
    assert x.shape == (288, 22, 500)
    assert meta.trial_id.is_unique
    assert meta.cue_sample.iloc[0] == 500
    assert meta.cue_sample.iloc[47] == 1440
    assert (meta.run <= 5).sum() == 240 and (meta.run == 6).sum() == 48
    assert np.array_equal(meta.groupby("label").size().to_numpy(), [72] * 4)
    runs[0] = dict(task, fs=125)
    savemat(fixture, {"data": runs}, do_compression=True)
    try:
        read_session(fixture, 1, "T")
    except ValueError:
        pass
    else:
        raise AssertionError("Malformed sampling rate was silently accepted")
del recording, runs, x, meta
print("MATLAB loader and split checks passed.")

## 7. Fit and lock every source model
Only session-1 training and validation recordings are used. Completed participants resume
after a disconnect without retuning. Each final model remains fitted on the 240 training trials.

In [ ]:
STUDY_ROOT = create_study(WORK_DIR, SUBJECTS, ENGINE_SOURCE, REFERENCE_CSV)
print("Results:", STUDY_ROOT)
select_all(SUBJECTS, CACHE_DIR, STUDY_ROOT)

## 8. Evaluate session 2 after locking choices
Apply the frozen source scaler, spatial filters, feature selector and classifier.
Target labels are used only for scoring. All 288 predictions, artifact flags and trial IDs
are exported for each participant.

In [ ]:
scores = evaluate_all(SUBJECTS, CACHE_DIR, STUDY_ROOT)
display(scores)

## 9. Summarize, plot, and export
Participants are the sampling units. The paired contrast is FBCSP BA minus the original
five-seed EEGNet mean for the same participant. This is a descriptive additional comparison,
not a replacement for the primary SAST result.

- summary.csv: mean BA, participant intervals and paired contrast.
- participant_scores.csv: participant scores and selected settings.
- paired_eegnet_comparison.csv: seed-averaged EEGNet scores and participant differences.
- paper_table_row.txt: created only after all nine participants finish; matches the original four table columns (nonspiking BA, raw spikes, calibrated spikes).
- subject folders: source split manifests, target predictions, fitted models and checksums.
- protocol.json, locked_selection.json, fbcsp_engine.py: reproducibility record.
- ZIP: outputs and fitted models; excludes raw EEG.

Keep the current manuscript unchanged until the results have been inspected.
This notebook supplies the classical baseline only. It does not add a second dataset
or an external SNN comparison.

In [ ]:
paired, summary, ARCHIVE = report_results(scores, REFERENCE_CSV, STUDY_ROOT)
display(summary)
display(paired)
if len(scores) == 9:
    print((STUDY_ROOT / "paper_table_row.txt").read_text())
print("Results folder:", STUDY_ROOT)
print("Downloadable ZIP:", ARCHIVE)

Download the ZIP from Drive or the notebook file browser.
In Colab, you can also uncomment and run the following cell.

In [ ]:
# from google.colab import files
# files.download(str(ARCHIVE))